# Condition Number & Numerical Health — Before, Principle, How-to, Effects

Models with misaligned coefficient orders amplify rounding errors, causing solvers to wander (numerically "ill-conditioned"). The max/min ratio of coefficients is not an accurate indicator of ill-conditioning just by looking at the width of the range. The true condition number $\kappa(A)$ is obtained by $\sigma_{\max}/\sigma_{\min}$ from the SVD of the coefficient matrix.

This notebook traces this phenomenon and countermeasures in the following pattern:

1. **Before (Issue)** — Measure $\kappa(A)$ in a model with poor unit choices and confirm the ill-conditioning.
2. **Principle** — See visually the spread of coefficient scales and the distribution of singular values.
3. **How-to (Application)** — Lower $\kappa(A)$ by changing the units of variables (scaling).
4. **Effects (Before/After)** — Compare solving stability before and after rescaling using `matrix_condition`, `scip_basis_condition`, and `mk.compare_variants`.

There are three subjects. (1) **A newly built model in this notebook** (a blending plan of raw materials in mg and high-grade additives) showing how unit choice worsens $\kappa(A)$. (Per the policy in CLAUDE.md, a minimal verifiable model is built from scratch when existing samples lack the right conditions). (2) As empirical backing, we also check how $\kappa(A)$ changes with loose/tight Big-M in a **production plan with fixed costs** (`samples/others/fixed_charge.py`, the same subject as `03_bigm_indicator.ipynb`) (measured in `experiments/run_condition.py`). (3) To provide a real example where the SCIP LP basis $\kappa$ actually becomes large, we use **Unit Commitment** (`samples/scheduling/unit_commitment.py`) (because in small toy models the basis $\kappa$ becomes unmeasurable/trivial, obscuring the difference from the static $\kappa(A)$).

**This notebook focuses on the condition number $\kappa(A)$ and addresses whether the countermeasure of rescaling is effective.** To see the coefficient scale as part of the diagnosis alongside block structures and symmetry detection, refer to [diagnose/04_static_diagnosis.ipynb](../diagnose/04_static_diagnosis.html).

In [ ]:
import sys, pathlib
# リポジトリルート(pyproject.toml を持つ階層)を探して import パスに載せる。
# docs/samples/ が存在するため "samples" 有無では docs で止まる。pyproject.toml を目印にする。
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/others", "samples/scheduling"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):  # 静的サイトに埋め込める形でグラフを表示(CDN の plotly.js を読む)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
from minlpkit.collectors.static_diag import matrix_condition, scip_basis_condition
import fixed_charge as fc
import unit_commitment as uc

# dataviz 参照パレット(minlpkit.live.plots と統一)
C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

## 1. Before (Issue) — $\kappa(A)$ of a model with poor unit choices

Consider a minimum-cost blending plan that mixes raw materials and high-grade additives to meet nutritional and energy standards.
Because the raw materials are handled in **mg units** (due to very precise lot management), the nutritional and energy contents are extremely small coefficients (0.03 to 0.05) "per 1 mg." On the other hand, the additives are in units of items (0 to 200 items), and their content coefficients range from 20 to 50, differing by orders of magnitude. This mixing of units ("mg vs items") separates the coefficient **column** of the raw materials by orders of magnitude from the other coefficient columns, making the matrix ill-conditioned.

In [ ]:
from pyscipopt import Model

def blend_model(rescale: bool) -> Model:
    # 原材料(rescale=False: mg単位/True: g単位)+ 添加剤カプセルの配合計画。
    # 数学的には等価(単位変換のみ)。rescale=True は x を 1000分の1(mg->g)にした変数。
    m = Model(); m.hideOutput()
    if not rescale:
        # 原材料 x: mg単位、0 <= x <= 3,000,000 (=3kg)。含有量は「1mgあたり」で極小
        x = m.addVar(lb=0, ub=3_000_000, name="x_mg")
        cost_x, nutrient_x, energy_x = 0.00006, 0.05, 0.03
    else:
        # 原材料 x: g単位、0 <= x <= 3,000。含有量は「1gあたり」= 1mgあたり*1000
        x = m.addVar(lb=0, ub=3_000, name="x_g")
        cost_x, nutrient_x, energy_x = 0.06, 50.0, 30.0
    y = m.addVar(vtype="I", lb=0, ub=200, name="capsules")
    m.addCons(nutrient_x * x + 50 * y >= 8000, name="nutrient")
    m.addCons(energy_x * x + 20 * y >= 4000, name="energy")
    m.setObjective(cost_x * x + 1500 * y, "minimize")
    m.data = dict(x=x, y=y, rescale=rescale)
    return m

kappa_raw = matrix_condition(blend_model(False))
kappa_res = matrix_condition(blend_model(True))
print(f"raw   (mg単位): kappa(A) = {kappa_raw['kappa']:.3e}  shape={kappa_raw['shape']}")
print(f"rescale(g単位): kappa(A) = {kappa_res['kappa']:.3e}  shape={kappa_res['shape']}")

Leaving it in mg units keeps $\kappa(A)$ large (ill-conditioned). Just by changing to g units (unit conversion = linear column scaling), the coefficient column of the raw materials (x) aligns with the order of magnitude of the additive's coefficient column (y), improving $\kappa(A)$.
Next, we view this principle via the coefficient distribution and the singular value spectrum.

## 2. Principle — Spread of coefficient scales and singular values

The condition number $\kappa(A)=\sigma_{\max}/\sigma_{\min}$ is the ratio of the maximum to the minimum singular value when performing SVD on the coefficient matrix $A$.
When the orders of coefficients are scattered, the scale of the singular values also scatters, making the ratio likely to be large (however, it does not exactly match the max/min coefficient ratio itself — it is strictly the stretching factor of the entire matrix).

In [ ]:
from minlpkit.collectors.static_diag import extract_coefficients

def build_A(rescale):
    m = blend_model(rescale)
    vars_ = m.getVars()
    vidx = {v.name: i for i, v in enumerate(vars_)}
    rows = []
    for c in m.getConss():
        if not c.isLinear():
            continue
        row = np.zeros(len(vars_))
        for vn, coef in m.getValsLinear(c).items():
            row[vidx[vn]] = coef
        rows.append(row)
    return np.array(rows)

A_raw, A_res = build_A(False), build_A(True)
sv_raw = np.linalg.svd(A_raw, compute_uv=False)
sv_res = np.linalg.svd(A_res, compute_uv=False)

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
    subplot_titles=("係数の絶対値レンジ(対数)", "特異値スペクトル σ(対数)"))

coefs_raw = np.abs(A_raw[A_raw != 0])
coefs_res = np.abs(A_res[A_res != 0])
fig.add_trace(go.Box(y=coefs_raw, name="raw (mg単位)", marker_color=C["muted"],
    boxpoints="all", jitter=0.5), row=1, col=1)
fig.add_trace(go.Box(y=coefs_res, name="rescale (g単位)", marker_color=C["s1"],
    boxpoints="all", jitter=0.5), row=1, col=1)

fig.add_trace(go.Bar(x=["sigma_max", "sigma_min"], y=[sv_raw[0], sv_raw[-1]],
    marker_color=C["muted"], name="raw", showlegend=False,
    text=[f"{v:.2e}" for v in [sv_raw[0], sv_raw[-1]]], textposition="outside"), row=1, col=2)
fig.add_trace(go.Bar(x=["sigma_max", "sigma_min"], y=[sv_res[0], sv_res[-1]],
    marker_color=C["s1"], name="rescale", showlegend=False,
    text=[f"{v:.2e}" for v in [sv_res[0], sv_res[-1]]], textposition="outside"), row=1, col=2)

fig.update_yaxes(type="log", gridcolor=C["grid"], linecolor=C["axis"], row=1, col=1)
fig.update_yaxes(type="log", gridcolor=C["grid"], linecolor=C["axis"], row=1, col=2)
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=420,
    margin=dict(l=50, r=20, t=56, b=40),
    title=dict(text="係数レンジと特異値スペクトル: raw(mg単位) vs rescale(g単位)",
               x=0.01, font=dict(color=C["ink"], size=15)))
show(fig)

In the raw (mg units, gray), the raw material coefficients (0.03 to 0.05) are separated by nearly 3 orders of magnitude from the additive coefficients (20 to 50), and since the minimum singular value is extremely small, $\kappa(A)$ worsens. In the rescale (g units, blue), the raw material coefficients are multiplied by 1000, aligning with the same order of magnitude (20 to 50) as the additives, and the ratio of singular values also shrinks significantly. **You can see that simply changing units does not change the contents of the matrix (the problem itself), but greatly changes how numerically tractable it is.**

## 3. How-to (Application) — Rescaling (unit conversion)

We can separately measure the ill-conditioning of the formulation itself and the instability of the actually solved basis using `matrix_condition` (pre-solve, SVD-based) and `scip_basis_condition` (post-solve, SCIP LP basis's `getCondition()`). Let's first measure both with the default settings (with presolve).

In [ ]:
for rescale, label in [(False, "raw (mg単位)"), (True, "rescale (g単位)")]:
    m = blend_model(rescale)
    kappa = matrix_condition(blend_model(rescale))["kappa"]
    basis_k = scip_basis_condition(m)
    print(f"{label:16s}: kappa(A)={kappa:.3e}   SCIP基底kappa={basis_k:.3e}")

**An honest caveat**: This small 2-variable model can be solved just by presolving, so SCIP doesn't retain a final LP basis, causing `getCondition()` to return `1e+99`, meaning "unmeasurable."
The static $\kappa(A)$ looks at the formulation itself before presolve, so it can be measured, but the SCIP basis $\kappa$ is only meaningful when "some LP is actually solved to the end." We will disable presolve and check again.

In [ ]:
def _set_no_presolve(m):
    m.setParam("presolving/maxrounds", 0)  # presolveでLPが消えないようにする
    return m

for rescale, label in [(False, "raw (mg単位)"), (True, "rescale (g単位)")]:
    basis_k = scip_basis_condition(_set_no_presolve(blend_model(rescale)))
    print(f"{label:16s}: SCIP基底kappa(presolveなし) = {basis_k:.3e}")

When presolve is disabled, both base $\kappa \approx 1$ and are very small — because this 2-variable model has few variables and the optimal basis itself is simple (almost diagonal), there can be cases where **the actually solved basis is stable even if the static $\kappa(A)$ is bad**. The static $\kappa(A)$ and basis $\kappa$ are complementary metrics and do not always match.
As a real example where the basis $\kappa$ actually becomes large, let's look at **Unit Commitment** (`samples/scheduling/unit_commitment.py`, a larger MILP with many variables/constraints and time coupling).

In [ ]:
uc_kappa = matrix_condition(uc.build_model())["kappa"]
uc_basis = scip_basis_condition(uc.build_model())
print(f"unit_commitment: 静的kappa(A)={uc_kappa:.3e}   SCIP基底kappa={uc_basis:.3e}")

For unit_commitment, the basis $\kappa \approx 10^{11}$, which is extremely large, indicating a domain with actual numerical instability risks (a target where SCIP 10's exact rational MILP mode, etc., is effective). The "case where basis $\kappa$ actually becomes a problem" that was buried in the toy model can be confirmed here.

Let's also confirm that the optimal value and the solution itself do not change with unit conversion (if x in the mg version is converted to g, they match).

In [ ]:
m_raw = blend_model(False); m_raw.optimize()
m_res = blend_model(True); m_res.optimize()
x_raw, x_res = m_raw.data["x"], m_res.data["x"]
print(f"raw    : obj={m_raw.getObjVal():.4f}  x={m_raw.getVal(x_raw):.1f}mg  "
      f"y={m_raw.getVal(m_raw.data['y']):.0f}")
print(f"rescale: obj={m_res.getObjVal():.4f}  x={m_res.getVal(x_res):.4f}g "
      f"(={m_res.getVal(x_res)*1000:.1f}mg)  y={m_res.getVal(m_res.data['y']):.0f}")

The objective value, number of additives, and raw material quantity (after conversion) all match perfectly — rescaling is an equivalent transformation that solely improves numerical tractability.

## 4. Effects (Before/After) — Comparing solving stability

We compare the root dual bound, gap, and number of nodes using `mk.compare_variants` (since it's a small model, differences may be minor, but the improvement of $\kappa(A)$ itself is the essence). In addition, we align the measured differences in $\kappa(A)$ of the loose/tight Big-M from `fixed_charge` (same as `03_bigm_indicator.ipynb`) and `unit_commitment` (follow-up of `experiments/run_condition.py`) to show that in larger models, differences in condition number directly translate into ill-conditioning risks.

In [ ]:
df = mk.compare_variants(
    {"raw (mg単位)":   lambda: blend_model(False),
     "rescale (g単位)": lambda: blend_model(True)},
    time_limit=10)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

In [ ]:
names = ["blend raw\n(mg単位)", "blend rescale\n(g単位)",
         "fixed_charge\nloose Big-M", "fixed_charge\ntight Big-M", "unit_\ncommitment"]
kappas = [matrix_condition(blend_model(False))["kappa"],
          matrix_condition(blend_model(True))["kappa"],
          matrix_condition(fc.build_model("loose"))["kappa"],
          matrix_condition(fc.build_model("tight"))["kappa"],
          uc_kappa]
# SCIP基底kappa: presolveが解いてしまうtoy blendモデルは presolveなしで測る。それ以外は既定設定。
scip_ks = [scip_basis_condition(_set_no_presolve(blend_model(False))),
           scip_basis_condition(_set_no_presolve(blend_model(True))),
           scip_basis_condition(fc.build_model("loose")),
           scip_basis_condition(fc.build_model("tight")),
           uc_basis]

fig = go.Figure(layout=base_layout(
    "条件数 kappa(A) 比較 — 大きいほど悪条件(丸め誤差でソルバーが迷走)",
    "", "条件数(対数)", height=440))
fig.add_trace(go.Bar(x=names, y=kappas, name="静的 kappa(A)(係数行列, solve前)",
    marker_color=C["s1"], text=[f"{k:.1e}" for k in kappas], textposition="outside",
    hovertemplate="%{x}<br>kappa(A)=%{y:.2e}<extra></extra>"))
fig.add_trace(go.Bar(x=names, y=scip_ks, name="SCIP LP基底 kappa(solve後)",
    marker_color=C["s2"], text=[f"{k:.1e}" for k in scip_ks], textposition="outside",
    hovertemplate="%{x}<br>基底kappa=%{y:.2e}<extra></extra>"))
fig.add_hline(y=1e7, line=dict(color=C["warn"], width=2, dash="dash"),
              annotation_text="数値的に要注意 (1e7)", annotation_position="top left",
              annotation_font=dict(color=C["warn"], size=11))
fig.update_yaxes(type="log")
fig.update_layout(barmode="group")
show(fig)

In [ ]:
print(f"blend        : raw kappa(A)={kappas[0]:.2e} -> rescale kappa(A)={kappas[1]:.2e} "
      f"({kappas[0]/kappas[1]:.0f}倍改善)")
print(f"fixed_charge : loose kappa(A)={kappas[2]:.2e} -> tight kappa(A)={kappas[3]:.2e} "
      f"({kappas[2]/kappas[3]:.0f}倍改善)")
print(f"unit_commitment: 静的kappa(A)={kappas[4]:.2e}   SCIP基底kappa={scip_ks[4]:.2e}"
      f"(数値的に要注意ラインの1e7を大きく超える)")

## Summary

- **Just by unit conversion (rescaling), $\kappa(A)$ improves by orders of magnitude.** Mathematically, this is a completely equivalent transformation (the optimal value and solution match), and the cost is nearly zero.
- Tightening Big-M (`03_bigm_indicator.ipynb`) is also, in a broad sense, a kind of rescaling that "aligns coefficient scales," and both are different approaches to addressing the same problem (ill-conditioning).
- **The static $\kappa(A)$ and SCIP basis $\kappa$ may not always match.** In simple models with few variables, the optimal basis can be stable even if the static $\kappa(A)$ is bad (toy example in section 3); conversely, when scale and time coupling are involved, as in unit_commitment, only the basis $\kappa$ may worsen drastically. Only by measuring both can you separately diagnose "the quality of the formulation" and "actual solving stability."

### Why SCIP Doesn't Do It Automatically

SCIP attempts internal scaling when solving LP relaxations (like `NORMALIZE`), but this is **numerical normalization at the row/column level** and does not step into the "units of variables" intended by the modeler.
Whether to use mg or g is a choice dependent on the model's semantics (what quantity the variable represents), and the solver cannot judge "which unit is correct" from the built coefficient matrix alone. That is why the modeler needs to be aware of it at the unit selection stage.

### When It Doesn't Work / Cautions

- Do not jump to the conclusion that there is a "numerical issue" based solely on the max/min ratio of coefficients. A natural cost difference (~1e3) is not a numerical problem; true ill-conditioning should be viewed as a range of 1e6 or more after presolve ([FINDINGS §1](../../playbook/08-condition.md)).
- Condition number diagnosis itself is a **tool for quantifying symptoms** and is not a solution on its own. Specific countermeasures include "reviewing units" and "eliminating Big-M" ([3. Eliminating Big-M](03_bigm_indicator.ipynb)).

Related: [Method Guide 8. Condition Number & Numerical Health](../../playbook/08-condition.md) /
[Method Guide 3. Eliminating Big-M](../../playbook/03-bigm.md) /
`experiments/run_condition.py` → [`condition.html`](../../gallery/condition.html).